# Grounding a Translator answer in public datasets — asthma

**Question.** NCATS Translator answers *"what might treat asthma?"* with `drug → gene → disease`
paths. Each `drug → gene` hop is a mechanistic claim. Can we find, in the NIAID Data Ecosystem
(NDE) and the repositories it indexes, primary data that speaks to those claims?

Four routes, differing in what they ask and of which data:

| route | question | source |
|---|---|---|
| **A** | does the drug change the gene's **expression**? | GXA differential-expression contrasts, via NDE `@type:Inference` |
| **B** | is there a **dataset** measuring expression after this drug? | GEO, discovered through NDE `@type:Sample` |
| **C** | what **other** compounds move this gene? | the same GXA contrasts, queried gene-first |
| **D** | does **activity** data support the claim? | PubChem BioAssay + ChEMBL |

Asthma is used throughout. Everything reads from archived artifacts under `data/ars/<pk>/`, so the
numbers are reproducible; ARS answers change over time and the archived `paths.json` is the
citable object.

In [1]:
import json, collections
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 60)
PK = "5b656c0f-b7da-4db4-ba1f-d3a794b422d4"      # ARS query id for asthma, 2026-09-01
ARS = Path("../data/ars") / PK

paths = json.loads((ARS / "paths.json").read_text())["paths"]
print(f"asthma (MONDO:0004979) — ARS pk {PK}")
print(f"{len(paths):,} drug→gene→disease paths")

asthma (MONDO:0004979) — ARS pk 5b656c0f-b7da-4db4-ba1f-d3a794b422d4
796 drug→gene→disease paths


## 1. What Translator returned

A creative-mode `biolink:treats` query on asthma, answered by 13 reasoning agents. We keep only
2-hop paths of the form `ChemicalEntity → Gene → Disease`, since those are the ones that name a
mechanism we could look for data about.

In [2]:
drugs = {p["drug_name"] for p in paths}
genes = {p["gene_name"] for p in paths}
edges = {(p["drug"], p["gene"]) for p in paths}
print(f"{len(drugs)} distinct drugs · {len(genes)} distinct genes · {len(edges)} distinct drug→gene edges")
print("contributing agents:", dict(collections.Counter(p["agent"] for p in paths)))

174 distinct drugs · 76 distinct genes · 220 distinct drug→gene edges
contributing agents: {'ars-ars-agent': 368, 'ara-unsecret': 117, 'ara-arax': 311}


In [3]:
# Top answers by the ARS score, with the gene(s) each path runs through.
best = {}
for p in paths:
    best[p["drug_name"]] = max(best.get(p["drug_name"], 0), p.get("score") or 0)

top10 = pd.DataFrame([
    {"drug": d, "score": round(s, 3),
     "genes": ", ".join(sorted({p["gene_name"] for p in paths if p["drug_name"] == d}))}
    for d, s in sorted(best.items(), key=lambda kv: -kv[1])[:10]
])
top10

,drug,score,genes
0,Terbutaline,0.994,"ADRB1, ADRB2"
1,Prednisolone,0.992,"NR3C1, NR3C2"
2,Adrenal Cortex Hormones,0.992,TNF
3,Prednisone,0.986,NR3C1
4,Zafirlukast,0.973,"CYSLTR1, CYSLTR2, MAPK1"
5,beclomethasone,0.971,NR3C1
6,Zileuton,0.969,ALOX5
7,Roflumilast,0.958,"PDE4A, PDE4B, PDE4D"
8,Triamcinolone,0.955,"CD1A, MMP1, NR3C1"
9,Betamethasone,0.949,NR3C1


The answer is pharmacologically sensible: β2-agonists (terbutaline), inhaled and systemic
corticosteroids (prednisolone, prednisone, beclomethasone, betamethasone, triamcinolone), a
leukotriene receptor antagonist (zafirlukast), a 5-lipoxygenase inhibitor (zileuton) and a PDE4
inhibitor (roflumilast) — each routed through the gene you would expect.

That is the set we now try to ground in data.

In [4]:
gene_counts = collections.Counter(p["gene_name"] for p in paths).most_common(8)
pd.DataFrame(gene_counts, columns=["gene", "paths through it"])

,gene,paths through it
0,P2RX3,164
1,NR3C1,90
2,ADRB2,56
3,HRH1,55
4,PTGS2,40
5,TNF,31
6,PDE4A,26
7,IL4R,26


## 2. Route A — does the drug change the gene's expression?

NDE's staging index carries `@type:Inference`: one record per Gene Expression Atlas
differential-expression contrast, with a log2 fold change, an adjusted p-value, and Biolink-typed
direction and aspect qualifiers. That lines up directly with a Translator drug→gene edge, so the
question can be stronger than *"does data exist?"* — it can be *"does the measured direction agree
with the asserted one?"*

Run **forward**: take Translator's edges and ask GXA about each one.

In [5]:
route_a = json.loads((ARS / "route_a.json").read_text())["results"]
verdicts = collections.Counter(r["verdict"] for r in route_a)

# 97 of the 220 edges name a research compound or synthetic peptide rather than a drug
# (IUPAC strings, "H-D-Phe-His-Leu-Leu-Arg-…"); free-text search cannot match those.
print(f"edges evaluated: {len(route_a)}")
for v, n in verdicts.most_common():
    print(f"  {v:24s} {n}")

edges evaluated: 123
  no_drug_data             116
  tested_not_significant   6
  agrees                   1


In [6]:
informative = [r for r in route_a if r["verdict"] != "no_drug_data"]
pd.DataFrame([
    {"drug": r["drug"], "gene": r["gene"], "verdict": r["verdict"],
     "contrasts": r["n_contrasts"], "median log2FC": r["median_log2fc"],
     "GXA experiment": ", ".join(r["experiments"])}
    for r in informative
])

,drug,gene,verdict,contrasts,median log2FC,GXA experiment
0,Prednisolone,NR3C1,tested_not_significant,0,NaN,
1,Prednisolone,NR3C2,tested_not_significant,0,NaN,
2,Prednisone,NR3C1,tested_not_significant,0,NaN,
3,Dupilumab,IL4R,tested_not_significant,0,NaN,
4,Sorafenib,ATG5,tested_not_significant,0,NaN,
5,Rifampicin,PPARGC1A,tested_not_significant,0,NaN,
6,Cyclic AMP,PPARGC1A,agrees,2,1.85,E-MTAB-2602


**Of 123 evaluable drug→gene edges, 7 returned any GXA drug-perturbation data, and exactly 1 had
expression data that agreed with the assertion.**

The `tested_not_significant` rows are informative in their own right: GXA *does* hold contrasts for
those drugs, measured genome-wide, and the gene never came up as differentially expressed — weak
evidence *against* the edge rather than absence of evidence. (`n_contrasts` above counts contrasts
for that drug–gene pair; the verdict rests on a separate count across all genes.) Four are clean
dose-controlled designs — `prednisolone 2 micromolar` in ALL cell lines, `dupilumab 150 milligram`
in skin lesions, `Sorafenib 5 micromolar` in HuH-7, `rifampicin 5 micromolar`.

⚠️ **`Prednisone → NR3C1` is a false positive**, so the honest count is 6 with data, not 7. All of
prednisone's human GXA contrasts come from one design —
`'systemic-onset juvenile idiopathic arthritis; Prednisone, NSAID, methotrexate' vs 'normal; none'`
— where the variable is *disease* and prednisone only describes what the patients were taking. The
arm rule misses it because the drug is a standalone factor absent from the reference arm; it cannot
distinguish a drug administered from a drug naming the cohort when the control is untreated healthy
subjects.

The single agreement is `Cyclic AMP → PPARGC1A`, and cyclic AMP is a second messenger rather than a
therapeutic.

Note which drugs are **absent entirely**. Asthma's first-line therapies have no human GXA contrasts
whatsoever — this is a selection gap in Expression Atlas, not a failure of the matching.

In [7]:
# Test-arm contrast counts measured directly against NDE staging, any species.
gxa_coverage = pd.DataFrame([
    ("budesonide", 0), ("formoterol", 0), ("fluticasone", 0), ("montelukast", 0),
    ("salbutamol / albuterol", 6), ("theophylline", 222), ("aspirin", 213),
    ("prednisolone", 481), ("dexamethasone", 57679),
], columns=["drug", "GXA contrasts (any species)"])
gxa_coverage["human"] = [0, 0, 0, 0, 6, 0, 0, 232, 23314]
gxa_coverage

,drug,GXA contrasts (any species),human
0,budesonide,0,0
1,formoterol,0,0
2,fluticasone,0,0
3,montelukast,0,0
4,salbutamol / albuterol,6,6
5,theophylline,222,0
6,aspirin,213,0
7,prednisolone,481,232
8,dexamethasone,57679,23314


Budesonide, formoterol, fluticasone and montelukast: **zero contrasts in any species**. Salbutamol's
six are incidental name matches. Theophylline's and aspirin's are rat toxicology. Dexamethasone is
the outlier that makes the atlas look better stocked than it is — it is a cell-culture workhorse.

So Route A cannot speak to asthma's actual pharmacology. Route B asks the weaker but more useful
question: does a dataset exist at all?

## 3. Route B — is there a dataset that profiles these drugs?

Drop the requirement that the data adjudicate a specific edge, and just ask: for a drug Translator
proposes, has anyone run an expression experiment on it?

Translator proposes **budesonide** and **formoterol** for asthma, each with gene edges:

In [8]:
for drug in ["Budesonide", "Formoterol"]:
    rows = [p for p in paths if p["drug_name"] == drug]
    g = sorted({p["gene_name"] for p in rows})
    print(f"{drug:12s} {len(g):2d} gene edges: {', '.join(g)}")

Budesonide   10 gene edges: ANXA1, CRHR1, CYP1A2, CYP2C19, CYP2C9, CYP3A4, EDN1, NR1I2, NR3C2, PGR
Formoterol    2 gene edges: ADRB1, CYP2C19


Now ask NDE. The useful signal is drug mentions at the **sample** level — a dataset-level mention
can come from an abstract, whereas a per-sample mention usually marks a treatment arm.

In [9]:
import sys
sys.path.insert(0, "../src")
from translator_nde.nde import NDEClient, PROD

nde = NDEClient(base_url=PROD)
GSE = "GSE162120"
print(f"{GSE} in NDE production:")
print(f"  Dataset records: {nde.count(f'identifier:\"{GSE}\"')}")
print(f"  Sample records : {nde.count(f'@type:Sample AND isBasisFor.identifier:\"{GSE}\"')}")
rec = next(iter(nde.scroll(f'identifier:\"{GSE}\"', fields="name,includedInDataCatalog.name", max_records=1)))
print(f"  title          : {rec['name']}")

GSE162120 in NDE production:


  Dataset records: 1
  Sample records : 118


  title          : The DISARM study: effects of inhaled corticosteroids on bronchial epithelial cell gene expression in COPD


In [10]:
import re
# NDE keeps the timepoint as a structured field but not the treatment arm, which survives only
# inside the free-text `sampleProcess` blob. A regex recovers it; the counts match GEO exactly.
arms, timepoints = collections.Counter(), collections.Counter()
for s in nde.scroll(f'@type:Sample AND isBasisFor.identifier:"{GSE}"',
                    fields="sampleProcess,temporalCoverage", max_records=200):
    m = re.search(r"\b(FOR/BUD|SAL/FLU|FOR)\b", s.get("sampleProcess") or "")
    arms[m.group(1) if m else "unlabelled"] += 1
    t = s.get("temporalCoverage") or {}
    timepoints[(t[0] if isinstance(t, list) else t).get("duration")] += 1

print("treatment arms:", dict(arms))
print("timepoints    :", dict(timepoints))

treatment arms: {'SAL/FLU': 40, 'FOR/BUD': 36, 'FOR': 41, 'unlabelled': 1}
timepoints    : {'pre-treatment': 62, 'post-treatment': 56}


**This is the bridge closing** — with one honest qualification, visible in the title printed above.

The DISARM trial randomised patients to formoterol (FOR), formoterol/**budesonide** (FOR/BUD) or
salmeterol/fluticasone (SAL/FLU), with bronchial brushings before and after 12 weeks and a counts
matrix deposited. Two of its three arms are drugs Translator proposed for asthma, and NDE indexes
the dataset and all 118 samples.

⚠️ **It is a COPD cohort, not an asthma cohort.** The drugs, the tissue and the delivery route are
the ones Translator's asthma answer names, and the two conditions overlap clinically, but this is
not the same indication. It is the right *drug* evidence in an adjacent disease — which is
typically what dataset discovery returns, and worth being explicit about rather than counting as a
clean hit.

It is also the only dataset of its kind we found: across the drug-treatment series re-analysed for
the other diseases in this project, none tested a drug Translator had proposed for that disease.

Two further limitations:

- the arm labels are not a structured NDE field — they had to be scraped from the free-text
  `sampleProcess` blob, so the join is fragile;
- all twelve budesonide/formoterol edges carry `direction: None`, so this dataset can support a
  coverage test, not a direction test.

## 4. Route C — what other compounds move these genes?

Invert the query. Translator's top asthma answers route through **NR3C1** (the glucocorticoid
receptor, for every corticosteroid) and **ADRB2** (for the β2-agonists). If moving those genes is
therapeutically useful, GXA can be asked gene-first which *other* compounds move them — and every
hit Translator did not propose is a repurposing hypothesis.

Querying gene-first plays to GXA's one structured axis: `observationAbout` carries a gene symbol
and an Ensembl id, so no text matching is needed on the gene side.

In [11]:
route_c = json.loads((ARS / "route_c.json").read_text())
spec = {c["compound"]: c for c in route_c["compounds"]}

def alternates(gene, n=5):
    rows = [r for r in route_c["contrasts"] if r["gene"] == gene]
    best = {}
    for r in rows:                       # keep each compound's largest effect
        k = r["compound"]
        if k not in best or abs(r["log2fc"] or 0) > abs(best[k]["log2fc"] or 0):
            best[k] = r
    out = [r for r in best.values() if spec[r["compound"]]["kind"] == "compound"]
    out.sort(key=lambda r: -(spec[r["compound"]]["specificity"] or 0))
    return pd.DataFrame([
        {"gene": gene, "compound": r["compound"], "direction": r["goal_direction"],
         "log2FC": r["log2fc"], "GXA experiment": r["experiment"],
         "genes this compound moves": spec[r["compound"]]["gxa_genes_moved"],
         "specificity": spec[r["compound"]]["specificity"]}
        for r in out[:n]])

pd.concat([alternates("NR3C1"), alternates("ADRB2")], ignore_index=True)

,gene,compound,direction,log2FC,GXA experiment,genes this compound moves,specificity
0,NR3C1,Digoxin,Upregulated,1.1,E-MTAB-5982,1000,0.0080
1,NR3C1,docetaxel,Upregulated,1.2,E-GEOD-28784,1000,0.0080
2,NR3C1,5-Aza,Upregulated,2.1,E-GEOD-41364,1000,0.0060
3,NR3C1,retinoic acid,Upregulated,1.2,E-MEXP-3577,1000,0.0060
4,NR3C1,paclitaxel,Upregulated,1.4,E-GEOD-28784,1000,0.0040
5,ADRB2,doxycycline,Downregulated,-5.1,E-GEOD-60548,1000,0.0400
6,ADRB2,valproic acid,Upregulated,3.9,E-MTAB-5984,1000,0.0380
7,ADRB2,trichostatin A,Upregulated,3.2,E-GEOD-37376,1000,0.0260
8,ADRB2,4-hydroxy-3-methyl-but-2-enyl pyrophosphate,Downregulated,-2.3,E-MEXP-1601,1000,0.0190
9,ADRB2,parthenolide,Downregulated,-1.6,E-GEOD-7538,381,0.0184


**Read this cautiously — the yield is poor, and the reason is instructive.**

Ranking by "how many of the disease's genes does this compound move" puts doxycycline, valproic
acid and trichostatin A on top of every list. Doxycycline is the Tet-on induction agent in a large
number of experiments, not a therapeutic; valproic acid and trichostatin A are HDAC inhibitors that
move thousands of genes. They score highly because they move *everything*.

The `specificity` column corrects for that — what fraction of a compound's total GXA activity lands
on this disease's genes — and once applied, nothing rises above ~4%. The compounds that survive are
still mostly promiscuous.

A second limit: 70 of asthma's 76 genes carry **no direction qualifier** from Translator, so for
most genes we cannot say which way is therapeutically useful and have to report both. Route C is
hypothesis-generating at best, and on this disease it generates little.

## 5. Route D — does activity data support the claims?

Routes A–C all interrogate expression. But Translator's drug→gene edges mostly assert changes in
**activity**, and a β2-agonist or a PDE4 inhibitor does not move its target's transcript. So ask
assays that measure what the edges actually claim: PubChem BioAssay (screening and dose-response)
and ChEMBL (curated mechanism of action).

Neither source is indexed by NDE — its catalog has LINCS and ReframeDB but no ChEMBL, PubChem or
BindingDB. Route D therefore reaches outside NDE, which is itself a coverage gap worth reporting.

The join is exact: Translator emits `NCBIGene:154`, PubChem's `Target GeneID` column is `154`, and
Node Normalizer supplies the compound's PubChem CID and ChEMBL id. No text matching anywhere.

In [12]:
route_d = json.loads((ARS / "route_d.json").read_text())["results"]
v = collections.Counter(r["verdict"] for r in route_d)
measured = v["mechanism_agrees"] + v["mechanism_disagrees"] + v["binding_confirmed"] + v["measured_inactive"]
print(f"edges evaluated: {len(route_d)}")
for k, n in v.most_common():
    print(f"  {k:20s} {n}")
print(f"\nedges with a directly measured compound–target result: "
      f"{measured}/{len(route_d)} ({100*measured/len(route_d):.0f}%)")
print(f"  vs Route A on the same answer set: 1/123 (1%)")

edges evaluated: 220
  binding_confirmed    129
  not_tested           34
  mechanism_untyped    27
  no_compound_id       20
  no_activity_data     8
  measured_inactive    2

edges with a directly measured compound–target result: 131/220 (60%)
  vs Route A on the same answer set: 1/123 (1%)


In [13]:
# Activity evidence for Translator's top-scoring asthma drugs: how the interaction is
# typed (ChEMBL) and how strongly it was measured (PubChem BioAssay, plus ChEMBL's
# standardised pChEMBL). The assay type is chosen to match the mechanism -- EC50/AC50
# for an agonist, IC50/Ki for an inhibitor -- since a single "most potent" value across
# all types can hand back a counter-screen.
from translator_nde.activity import PubChemBioAssay, preferred_potency, pchembl_to_um

pubchem = PubChemBioAssay("../data/activity")

def fmt(um):
    if um is None:
        return "–"
    return f"{um*1000:.3g} nM" if um < 1 else f"{um:.3g} µM"

mech = [r for r in route_d if r["chembl_action_type"]]
rows = []
for r in sorted(mech, key=lambda r: -(r["max_phase"] or 0))[:10]:
    by_type = pubchem.potency_by_type(r["drug_cid"], r["gene"]) if r["drug_cid"] else {}
    pref = preferred_potency(by_type, r["chembl_action_type"])
    rows.append({
        "drug": r["drug_name"], "gene": r["gene_name"],
        "action": r["chembl_action_type"],
        "assay": pref[0] if pref else "–",
        "PubChem": fmt(pref[1] if pref else None),
        "ChEMBL (pChEMBL)": fmt(pchembl_to_um(r["pchembl_max"])),
        "active/inactive": f"{r['n_active']}/{r['n_inactive']}",
    })
pd.DataFrame(rows)

,drug,gene,action,assay,PubChem,ChEMBL (pChEMBL),active/inactive
0,Terbutaline,ADRB2,AGONIST,EC50,3.16 µM,2.51 µM,5/0
1,Prednisolone,NR3C1,AGONIST,EC50,31.6 nM,0.525 nM,14/0
2,Prednisone,NR3C1,AGONIST,AC50,268 nM,28.8 nM,8/7
3,Zafirlukast,CYSLTR1,ANTAGONIST,IC50,0.26 nM,0.257 nM,13/0
4,Zileuton,ALOX5,INHIBITOR,IC50,150 nM,151 nM,94/0
5,Roflumilast,PDE4B,INHIBITOR,–,–,0.151 nM,1/0
6,Roflumilast,PDE4A,INHIBITOR,IC50,0.21 nM,0.209 nM,8/0
7,Triamcinolone,NR3C1,AGONIST,AC50,9.1 nM,9.12 nM,8/1
8,Mepolizumab,IL5,INHIBITOR,–,–,–,0/0
9,Aspirin,PTGS2,INHIBITOR,IC50,2.4 µM,2.4 µM,10/2


**This is the same drug list Route A drew a blank on.** Terbutaline, prednisolone, zafirlukast,
zileuton and roflumilast have no usable expression evidence and complete activity evidence —
approved drugs, correctly typed, with measured potencies.

The two sources agree where both have data, which is a useful check on the join: zileuton/ALOX5
0.15 vs 0.151 µM, roflumilast/PDE4A 0.21 vs 0.209 nM, aspirin/PTGS2 2.4 vs 2.399 µM,
triamcinolone/NR3C1 9.1 vs 9.12 nM. They are independent records reached by different identifiers —
PubChem via the NCBI Gene id, ChEMBL via the UniProt accession — so the agreement says the compound
and target were resolved correctly on both sides.

The blanks are informative too. Mepolizumab and dupilumab are monoclonal antibodies: no PubChem
CID, no small-molecule assay, so the row is empty by construction rather than by absence of
evidence. PubChem also records **`Inactive`** outcomes, which the expression routes structurally
cannot — GXA stores only significant results, so a missing record is uninformative, whereas a
recorded inactive measurement is a real negative.

## Summary

| route | asks | asthma result |
|---|---|---|
| **A** | does the drug change the gene's expression? | **1 / 123 edges** agreed; 7 had any data. Asthma's first-line drugs are absent from GXA entirely. |
| **B** | is there a dataset profiling this drug? | **GSE162120** — the DISARM trial, budesonide and formoterol arms, 118 samples, all indexed in NDE (a COPD cohort, not asthma). |
| **C** | what other compounds move these genes? | 73 compounds surfaced, but specificity ≤4%; dominated by HDAC inhibitors and the Tet-on inducer. |
| **D** | does activity data support the claim? | **131 / 220 edges (60%)** measured; 27 curated mechanisms recovering Translator's top answers exactly. |

**What the exercise shows.** The bridge is real but it does not run where the original sketch
assumed. Expression data — precomputed (A) or discovered (B) — cannot adjudicate most Translator
edges, because those edges assert *activity* and the atlases measure *abundance*; and because
Expression Atlas simply does not contain the drugs clinicians use for asthma. Activity data answers
the question the edges actually ask, at fifty times the coverage, but sits outside NDE.

The one place all three pieces line up — a Translator-proposed drug, a public dataset that perturbs
with it, and NDE indexing that dataset — is GSE162120, and even there the cohort is COPD rather
than asthma. Making that case routine rather than exceptional is the work.